Rewrite of script by same name in notebook format, for testing gurobipy code etc. 

In [60]:
import numpy as np
import time
import os
from pathlib import Path
import corono as coro
from astropy.io import fits

# Parameters

Telescope name

In [61]:
pupil_name   = 'HiCAT'
problem_name = 'MaxTau'
solver       = 'stdgrb' #,'stdgrb' #  'gurobipy', 'scipy.linprog'

In [62]:
slvLogToConsole = 0
slvCrossover    = 0
slvMethod       = 2
allLogToConsole = 0

In [63]:
MinIsland   = False
Binarity    = False
FirstDerGlobalLim = 100.
BinarityReg       = 0.1

nPup = corono0.params['nPup']

In [64]:
nPup = 486
nFPM = 50
Fmax2d = 16
nImg2d = 32

mask radius in lam0/D units

In [65]:
rMask = 8.543/2

dark zone bounds (inner and outer edges) in lam0/D unit

In [66]:
rho0 =  3.75
rho1 = 15.00

contrast in the dark region

In [67]:
cDarkHole = 8.0

tau (integrated Pupil transmission)

In [68]:
tau   = 0.4

CtrBtwnPix2

In [69]:
corono_name = 'APLC' # 'SP' or 'APLC'
CtrBtwnPix  = True
CtrBtwnPix2 = True
Pupil2dSym  = True

nlam

In [70]:
bw   = 0.1
nlam = 3

In [71]:
do_fits = True

# File reading for Pupil and Lyot stop

In [72]:
fdir = Path('./pupils/2D/').resolve()
fname_pup = 'HiCAT-Aper_F-N0486_Hex3-Ctr0972-Obs0195-SpX0017-Gap0004.fits'
fname_lys = 'HiCAT-Lyot_F-N0486_LS-Ann-gy-ID0345-OD0807-SpX0036.fits'

fpath_pup = fdir / fname_pup
fpath_lys = fdir / fname_lys
Pupil2d    = fits.getdata(fpath_pup)
LyotStop2d = fits.getdata(fpath_lys)

In [73]:
if solver != 'gurobipy' and solver != 'stdgrb':
    solver = 'scipy'

In [74]:
params = coro.to_dict(nPup=nPup, Fmax2d = Fmax2d, nImg2d=nImg2d, nFPM = nFPM,
                 rho0=rho0, rho1=rho1, cDarkHole=cDarkHole, tau=tau, 
                 CtrBtwnPix=CtrBtwnPix, CtrBtwnPix2 = CtrBtwnPix2,
                 nlam=nlam, bw=bw,
                 Pupil2d = Pupil2d, LyotStop2d = LyotStop2d,
                 Pupil2dSym = Pupil2dSym, rMask=rMask,
                 problem_name = problem_name, 
                 solver = solver, 
                 corono_name = corono_name, pupil_name = pupil_name,
                 slvLogToConsole = slvLogToConsole,
                 slvCrossover = slvCrossover, slvMethod = slvMethod,
                 allLogToConsole = allLogToConsole,
                 MinIsland = MinIsland, FirstDerGlobalLim = FirstDerGlobalLim,
                 Binarity = Binarity, BinarityReg = BinarityReg)

# Coronagraph defintion

In [75]:
if corono_name == 'SP':
    corono0 = coro.design.SP2d(**params)
elif corono_name == 'APLC':
    corono0 = coro.design.APLC2d(**params)
else:
    raise NameError('{0}: Not an existing coronagraph!'.format(corono_name))

# Problem definition

In [76]:
if problem_name == 'MaxTau':
    # Maximization of the integrated amplitude transmission of the apodizer
    problem1 = coro.optim_2d.MaxTau(corono=corono0, **params)
elif problem_name == 'MaxContrastL1':
    # Maximization of the contrast under L1-norm
    problem1 = coro.optim_2d.MaxContrast(corono=corono0, Lnorm='L1',**params)
elif problem_name == 'MaxContrastLinf':
    # Maximization of the contrast under L-infinite norm
    problem1 = coro.optim_2d.MaxContrast(corono=corono0, Lnorm='Linf',**params)
else:
    raise NameError('{0}: Not an existing optimization problem!'.format(problem_name))

# Apodizer solutions

In [77]:
t0 = time.time()
Apod1 = problem1.solve_model()
t1 = time.time()
print('optimization time             : {0:.2f}s'.format(t1-t0))

optimization time             : 688.34s


# Generation of full apodizer for quarter pupil optimization

In [78]:
Apod1_2d = np.reshape(Apod1, (corono0.nPup, corono0.nPup))

if Pupil2dSym == True:
        Apod1_2dtmp =  Apod1_2d[corono0.nPup//2:, corono0.nPup//2:]
        Apod1_2d[:corono0.nPup//2, corono0.nPup//2:] = np.flip(Apod1_2dtmp, axis=0)
        Apod1_2d[:, :corono0.nPup//2]          = np.flip(Apod1_2d[:, corono0.nPup//2:], axis=1)

# Save apodizer

In [79]:
fdir = Path('./results/2D/dat_pyth').resolve() / pupil_name
if not os.path.exists(fdir):
    os.makedirs(fdir)
    
fname = problem1.get_filename() + '.fits'
fpath = fdir / fname

if do_fits is True:
    fits.writeto(fpath, Apod1_2d, overwrite=True)